# M5v2 v2 Re-train (frames + ADE20K) — Colab T4

**목표**: mAP50 0.62 → **0.85+ 도전**

**변경**:
- 데이터: frames 4208장 → frames + ADE20K **약 17000장** (4배 증가)
- 모델: yolov11m (yolov8m 대비 +3~5)
- imgsz: 1280 (T4 16GB batch=4 또는 batch=8 with imgsz=960)
- lr: 1e-4 → 5e-4 (더 공격적)

**Drive 준비:**
1. `frames_ade.zip` (약 1~2GB)

**예상 시간:** 5~7시간

**런타임:** GPU T4

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
!pip install -q ultralytics onnx onnxslim onnxruntime-gpu

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os, shutil, zipfile
from pathlib import Path

# ★ Drive 폴더 (공유 받은 폴더 이름) ★
DRIVE_DIR = Path('/content/drive/MyDrive/drone_inspect_A')  # 본인 폴더 이름으로 변경
ZIP_NAME = 'frames_ade.zip'

WORK = Path('/content/m5v2_train')
WORK.mkdir(parents=True, exist_ok=True)

zip_path = DRIVE_DIR / ZIP_NAME
print(f'zip exists? {zip_path.exists()} -> {zip_path}')
assert zip_path.exists(), f'{zip_path} 없음'

ds_dir = WORK / 'frames_ade'
if not (ds_dir / 'images' / 'val').exists():
    if ds_dir.exists():
        shutil.rmtree(ds_dir, ignore_errors=True)
    print('Extracting...')
    with zipfile.ZipFile(zip_path) as z:
        for m in z.namelist():
            rel = m.replace('\\', '/')
            target = WORK / rel
            if rel.endswith('/'):
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(m) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
    print(f'Done: {ds_dir}')

for split in ['train', 'val', 'test']:
    p = ds_dir / 'images' / split
    if p.exists():
        print(f'  {split}: {len(list(p.glob("*")))} images')

In [ ]:
yaml_text = f"""path: {ds_dir}
train: images/train
val: images/val
test: images/test

nc: 4
names:
  0: wall_edge
  1: ceiling_edge
  2: door_frame
  3: window_frame
"""
data_yaml = WORK / 'frames_ade.yaml'
data_yaml.write_text(yaml_text)
print(data_yaml.read_text())

## Drive 자동 저장 + Resume 체크

In [ ]:
AUTOSAVE = DRIVE_DIR / 'm5v2_v2_autosave'
AUTOSAVE.mkdir(parents=True, exist_ok=True)
PROJECT = Path('/content/runs/m5v2_v2')
PROJECT.mkdir(parents=True, exist_ok=True)

RESUME = False
resume_pt = AUTOSAVE / 'last.pt'
if resume_pt.exists():
    print(f'Found previous: {resume_pt}')
    weights_dir = PROJECT / 'train' / 'weights'
    weights_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(resume_pt, weights_dir / 'last.pt')
    if (AUTOSAVE / 'best.pt').exists():
        shutil.copy2(AUTOSAVE / 'best.pt', weights_dir / 'best.pt')
    RESUME = True
else:
    print('Fresh start.')

## 학습 (yolov11m + 1280 + 50ep)

In [ ]:
from ultralytics import YOLO
import threading, time

stop_flag = [False]
def autosave_loop():
    while not stop_flag[0]:
        time.sleep(300)
        for src in [PROJECT/'train/weights/last.pt', PROJECT/'train/weights/best.pt']:
            if src.exists():
                try: shutil.copy2(src, AUTOSAVE / src.name)
                except: pass
        rcsv = PROJECT/'train/results.csv'
        if rcsv.exists():
            try: shutil.copy2(rcsv, AUTOSAVE / 'results.csv')
            except: pass
        print(f'[autosave] {time.strftime("%H:%M:%S")} -> {AUTOSAVE}')

t = threading.Thread(target=autosave_loop, daemon=True)
t.start()

if RESUME:
    model = YOLO(str(PROJECT / 'train/weights/last.pt'))
    train_kwargs = {'resume': True}
else:
    model = YOLO('yolo11m.pt')
    train_kwargs = {
        'data': str(data_yaml),
        'epochs': 50,
        'batch': 4,              # T4 16GB + 1280 → batch=4
        'imgsz': 1280,
        'cache': 'disk',
        'workers': 4,
        'optimizer': 'AdamW',
        'lr0': 5e-4,             # 더 공격적
        'lrf': 0.01,
        'cos_lr': True,
        'patience': 15,
        'warmup_epochs': 3,
        'close_mosaic': 15,
        'freeze': 0,
        'hsv_h': 0.015, 'hsv_s': 0.5, 'hsv_v': 0.4,
        'degrees': 3.0, 'translate': 0.1, 'scale': 0.3,
        'shear': 1.0, 'perspective': 0.0005,
        'flipud': 0.0, 'fliplr': 0.5,
        'mosaic': 1.0, 'mixup': 0.1, 'copy_paste': 0.3,
        'erasing': 0.0,
        'multi_scale': 0.2,
        'save_period': 5,
        'plots': True,
        'project': str(PROJECT.parent),
        'name': PROJECT.name,
        'exist_ok': True,
    }

results = model.train(**train_kwargs)
stop_flag[0] = True
print('Train done.')

In [ ]:
best_path = PROJECT / 'train/weights/best.pt'
best_model = YOLO(str(best_path))
best_model.export(format='onnx', opset=17, dynamic=True, simplify=True)
onnx_path = best_path.with_suffix('.onnx')

metrics = best_model.val(data=str(data_yaml), imgsz=1280, batch=4)
print(f'\n=== M5v2 v2 결과 ===')
print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
print(f'precision: {metrics.box.mp:.4f}')
print(f'recall:    {metrics.box.mr:.4f}')
print(f'0.9? {"YES ✅" if metrics.box.map50 >= 0.9 else "NO"}')

In [ ]:
OUT = DRIVE_DIR / 'm5v2_v2_results'
OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(onnx_path, OUT / 'm5_yolo_seg_frames.onnx')
shutil.copy2(best_path, OUT / 'm5v2_v2_best.pt')
if (best_path.parent.parent / 'results.csv').exists():
    shutil.copy2(best_path.parent.parent / 'results.csv', OUT / 'results.csv')
for plot in best_path.parent.parent.glob('*.png'):
    shutil.copy2(plot, OUT / plot.name)
print('Saved:', OUT)